# Audio Analysis in Google Drive using Classification Models

This notebook reads audio recordings directly from your **Google Drive** and runs a **Neural Network Classification Model** to detect specific species in each recording.

---

### What this notebook does (in order):
1. **Connects to your Google Drive** to read audio and save results
2. **Installs the necessary software** automatically
3. **Scans your audio folder** to discover recording sites and files
4. **Loads a model** from HuggingFace or your Google Drive (e.g. BirdNET, Perch, custom)
5. **Analyzes every recording** and saves the results as text files on your Drive

### Folder structure:
You can organize your audio files in two ways:

```
# Flat — all files in one folder (single site)
MyDrive/audio/
  20250101_180000.wav
  20250101_183000.wav

# Subfolders — each subfolder is a separate recording site
MyDrive/audio/
  POINT_A/
    20250101_180000.wav
  POINT_B/
    20250101_180000.wav
```

### Filename format:
The notebook reads timestamps from filenames. Supported formats:
- **AudioMoth** (default): `YYYYMMDD_HHMMSS.wav` — e.g. `20250615_203000.wav`
- **AudioMoth with prefix**: `PREFIX_YYYYMMDD_HHMMSS.wav` — e.g. `SM4_20250615_203000.wav`
- **ISO datetime**: `YYYY-MM-DD_HH-MM-SS.wav` — e.g. `2025-06-15_20-30-00.wav`

Files whose names do not match any pattern are still analyzed — the timestamp-based columns in results will be left blank.

### Before you start:
- You need a **Google account** with Google Drive
- You need a **TFLite (`.tflite`) or ONNX (`.onnx`) model file** and a matching **labels file** (`.txt`, one species per line)
- Your audio files must be on your Google Drive

### How to run:
Run each cell **one at a time**, from top to bottom. Click the ▶ button on the left of each cell, or press `Shift + Enter`.

> **New to notebooks?** A cell with `[ ]` on the left has not been run. After running, it shows `[1]`, `[2]`, etc. If you see an error (red text), read the message — it usually tells you exactly what to fix.

---

Created by [biodiversica](https://biodiversica.xyz). For issues, questions, or feedback, open an issue on [GitHub](https://github.com/biodiversica/bioacoustic-ipynbs/issues) or visit [biodiversica.xyz](https://biodiversica.xyz).

---
## Step 1 — Connect Google Drive & Install Software

Run the two cells below. The first will ask you to **allow access to your Google Drive** — click the link and follow the steps.

In [1]:
#@title 📂 Connect Google Drive { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive connected successfully.')

In [2]:
#@title 📦 Install packages { display-mode: "form" }

!pip install ai-edge-litert onnxruntime librosa soundfile huggingface_hub -q

print('\nAll packages installed successfully.')

---
## Step 2 — Configuration

Fill in the forms below. **You do not need to edit any code** — just type or select your values in each form and run the cell.

Run all three forms from top to bottom:
1. **General Settings** — confidence threshold, results folder
2. **Audio Input** — where your recordings are, how filenames are formatted, and an optional date/time filter
3. **Model** — where your model is stored and how it works

> **Tip:** The forms hide the code automatically. To see the underlying code, click the `{ }` icon in the top-right corner of any form cell.

In [3]:
#@title ⚙️ General Settings { display-mode: "form" }

import os

#@markdown **Results folder** — where detection results will be saved on your Google Drive.
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/classification_results"  #@param {type:"string"}

#@markdown ---
#@markdown **Detection threshold** — minimum confidence score to save a detection (0.0–1.0).
#@markdown Lower = more detections (may include false positives). Higher = fewer but more reliable.
MIN_CONFIDENCE = 0.5  #@param {type:"number"}
MIN_CONFIDENCE = min(max(MIN_CONFIDENCE, 0.0), 1.0)  # keep within 0.0–1.0

#@markdown ---
#@markdown **Top K detections per segment** — maximum number of detections to keep per audio segment.
#@markdown 1 = only the highest-confidence detection per segment. Increase to keep more.
TOP_K = 1  #@param {type:"integer"}

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print(f"Results folder  : {DRIVE_RESULTS_DIR}")
print(f"Min. confidence : {MIN_CONFIDENCE}")
print(f"Top K           : {TOP_K}")

In [4]:
#@title ⚙️ Audio Preprocessing { display-mode: "form" }
#@markdown Optional preprocessing applied to each recording before inference.
#@markdown
#@markdown ---
#@markdown **Frequency filter** — remove frequencies outside the range of interest.
FILTER_TYPE = "none"  #@param ["none", "lowpass", "highpass", "bandpass"]

#@markdown **Low-cut frequency (Hz)** — used for highpass and bandpass filters.
FILTER_LOW_HZ = 0  #@param {type:"integer"}

#@markdown **High-cut frequency (Hz)** — used for lowpass and bandpass filters.
FILTER_HIGH_HZ = 15000  #@param {type:"integer"}

#@markdown ---
#@markdown **Playback speed** — 1.0 = normal. Below 1.0 slows down and lengthens audio;
#@markdown above 1.0 speeds up and shortens it. Useful for shifting frequency content
#@markdown into the model's expected range (e.g. 0.5x halves all frequencies).
AUDIO_SPEED = 1.0  #@param {type:"number"}
AUDIO_SPEED = min(max(AUDIO_SPEED, 0.25), 4.0)  # keep within 0.25–4.0

_preprocess_lines = []
if FILTER_TYPE != 'none':
    if FILTER_TYPE == 'lowpass':
        _preprocess_lines.append(f'Filter   : lowpass <= {FILTER_HIGH_HZ} Hz')
    elif FILTER_TYPE == 'highpass':
        _preprocess_lines.append(f'Filter   : highpass >= {FILTER_LOW_HZ} Hz')
    elif FILTER_TYPE == 'bandpass':
        _preprocess_lines.append(f'Filter   : bandpass {FILTER_LOW_HZ}-{FILTER_HIGH_HZ} Hz')
if AUDIO_SPEED != 1.0:
    _preprocess_lines.append(f'Speed    : {AUDIO_SPEED}x')
if _preprocess_lines:
    print('Preprocessing enabled:')
    for _l in _preprocess_lines:
        print(f'  {_l}')
else:
    print('Preprocessing : none')

#@markdown ---
#@markdown **Time of day filter** — only analyze recordings within a specific daily time window.
#@markdown Leave disabled to analyze all recordings regardless of time.
TIME_FILTER_ENABLED = False  #@param {type:"boolean"}
#@markdown **Window start (HH:MM)**
TIME_FILTER_START = "06:00"  #@param {type:"string"}
#@markdown **Window end (HH:MM)** — can cross midnight, e.g. 22:00 to 06:00.
TIME_FILTER_END = "20:00"  #@param {type:"string"}

if TIME_FILTER_ENABLED:
    print(f'Time filter : {TIME_FILTER_START} – {TIME_FILTER_END}')

In [5]:
#@title 🗂️ Audio Input { display-mode: "form" }

#@markdown **Audio folder** — path to the folder on your Google Drive that contains the recordings.
#@markdown - If the folder has **subfolders**, each subfolder will be treated as a separate recording site.
#@markdown - If all files are directly in this folder, the folder name is used as the site name.
#@markdown Example: `/content/drive/MyDrive/my_project/audio`
DRIVE_INPUT_DIR = "/content/drive/MyDrive/audio"  #@param {type:"string"}

#@markdown ---
#@markdown **Filename prefix** — leave blank for standard AudioMoth filenames (`YYYYMMDD_HHMMSS.wav`).
#@markdown If your recorder adds a prefix before the timestamp, enter it here **without** the trailing underscore.
#@markdown Example: enter `SM4` for files named `SM4_20250615_203000.wav`
FILENAME_PREFIX = ""  #@param {type:"string"}

#@markdown ---
#@markdown **Date/time filter** (optional) — when enabled, only files whose filename timestamp
#@markdown falls within this range will be analyzed. Files with no parseable timestamp are always included.
#@markdown Leave disabled to analyze all files in the folder.
FILTER_ENABLED = False  #@param {type:"boolean"}
FILTER_START_DATE = "2025-01-01"  #@param {type:"date"}
FILTER_START_TIME = "00:00"  #@param {type:"string"}
FILTER_END_DATE = "2025-12-31"  #@param {type:"date"}
FILTER_END_TIME = "23:59"  #@param {type:"string"}

#@markdown ---
#@markdown **Site coordinates** (optional) — latitude and longitude per recording site.
#@markdown Format: `SITE_NAME=lat,lon` separated by semicolons.
#@markdown The site name must match the subfolder name exactly (or the root folder name if no subfolders).
#@markdown Example: `POINT_A=-15.1,-47.2;POINT_B=-16.3,-48.1`
SITE_COORDINATES = ""  #@param {type:"string"}

FILENAME_PREFIX = FILENAME_PREFIX.strip()

_latlon_map = {}
for _e in SITE_COORDINATES.split(';'):
    _e = _e.strip()
    if '=' in _e:
        _n, _c = _e.split('=', 1)
        _p = _c.split(',')
        try: _latlon_map[_n.strip()] = (float(_p[0]), float(_p[1]))
        except Exception: pass

if not os.path.isdir(DRIVE_INPUT_DIR):
    print(f"WARNING: Folder not found: {DRIVE_INPUT_DIR}")
    print("Check the path above — make sure Google Drive is connected and the folder exists.")
else:
    print(f"Audio folder : {DRIVE_INPUT_DIR}")
    if FILENAME_PREFIX:
        print(f"Filename prefix : '{FILENAME_PREFIX}'  (expects files like {FILENAME_PREFIX}_YYYYMMDD_HHMMSS.wav)")
    else:
        print("Filename prefix : none  (expects files like YYYYMMDD_HHMMSS.wav)")
    if FILTER_ENABLED:
        print(f"Date filter     : {FILTER_START_DATE} {FILTER_START_TIME} → {FILTER_END_DATE} {FILTER_END_TIME}")
    else:
        print("Date filter     : disabled (all files will be analyzed)")
    if _latlon_map:
        for _sn, (_la, _lo) in _latlon_map.items():
            print(f"  Coordinates   : {_sn} → lat={_la}, lon={_lo}")
    else:
        print("Site coordinates: not set")

In [6]:
#@title 🤖 Model { display-mode: "form" }

#@markdown **Model source** — where is your model file stored?
MODEL_SOURCE = "google_drive"  #@param ["huggingface", "google_drive"]

#@markdown ---
#@markdown ### If model source = Google Drive
#@markdown Full path to the model file on your Drive (`.tflite` or `.onnx`).
#@markdown Example: `/content/drive/MyDrive/models/frog_detector.tflite`
DRIVE_MODEL_PATH  = "/content/drive/MyDrive/Models/model.tflite"  #@param {type:"string"}
#@markdown Full path to the labels file on your Drive.
DRIVE_LABELS_PATH = "/content/drive/MyDrive/Models/labels.txt"  #@param {type:"string"}

#@markdown ---
#@markdown ### If model source = HuggingFace
#@markdown The repo ID is the part after `huggingface.co/` in the model URL.
#@markdown Default: `justinchuby/BirdNET-onnx` (BirdNET v2.4 in ONNX format)
HF_REPO_ID     = "justinchuby/BirdNET-onnx"  #@param {type:"string"}
HF_MODEL_FILE  = "model.onnx"               #@param {type:"string"}
#@markdown The labels file can be in a **different** HuggingFace repo. Leave blank to use the same repo as the model.
HF_LABELS_REPO = ""                          #@param {type:"string"}
HF_LABELS_FILE = "BirdNET_GLOBAL_6K_V2.4_Labels.txt"  #@param {type:"string"}

#@markdown ---
#@markdown ### Labels file settings
#@markdown **Has header row?** — check if the first line of the labels file is a column header (not a label).
LABELS_HAS_HEADER = False  #@param {type:"boolean"}
#@markdown **Label column index** — which column contains the label name? (0 = first column, 1 = second, etc.)
LABELS_COLUMN_INDEX = 0  #@param {type:"integer"}
#@markdown **Column delimiter** — how columns are separated in the labels file.
LABELS_DELIMITER = "underscore (_)"  #@param ["tab", "comma (,)", "semicolon (;)", "underscore (_)"]

#@markdown ---
#@markdown **Activation function** — how the model converts its output to confidence scores.
#@markdown `sigmoid` = each species scored independently (most common). `softmax` = scores sum to 1. `none` = model already outputs probabilities.
ACTIVATION = "sigmoid"  #@param ["sigmoid", "softmax", "none"]

#@markdown ---
#@markdown **Sigmoid sensitivity** — steepness of the sigmoid curve (only used when activation = `sigmoid`). `-1.0` = standard sigmoid; more negative = steeper.
SIGMOID_SENSITIVITY = -1.0  #@param {type:"number"}
#@markdown **Sigmoid bias** — horizontal shift of the sigmoid (only used when activation = `sigmoid`). `1.0` = no shift; above 1.0 = more sensitive (higher scores); below 1.0 = more conservative (lower scores).
SIGMOID_BIAS = 1.0  #@param {type:"number"}
SIGMOID_BIAS = min(max(SIGMOID_BIAS, 0.01), 1.99)  # keep within 0.01–1.99

#@markdown **Sample rate (Hz)** — audio sample rate the model expects.
#@markdown BirdNET = 48000 · Google Perch = 32000 · Custom: check your model documentation.
SAMPLE_RATE = 48000  #@param {type:"integer"}

#@markdown **Segment duration (seconds)** — length of each audio chunk fed to the model.
#@markdown BirdNET = 3.0 · Google Perch = 5.0 · Custom: check your model documentation.
SEGMENT_DURATION_S = 3.0  #@param {type:"number"}

#@markdown **Segment overlap (0.0–0.9)** — fraction of overlap between consecutive audio chunks.
#@markdown 0.0 = no overlap (default). 0.5 = 50% overlap (twice as many segments, better coverage).
SEGMENT_OVERLAP = 0.0  #@param {type:"number"}
SEGMENT_OVERLAP = min(max(SEGMENT_OVERLAP, 0.0), 0.9)  # keep within 0.0–0.9

print(f"Model source       : {MODEL_SOURCE}")
print(f"Activation         : {ACTIVATION}")
print(f"Sigmoid      : sensitivity={SIGMOID_SENSITIVITY}  bias={SIGMOID_BIAS}  (used only when activation = sigmoid)")
print(f"Sample rate        : {SAMPLE_RATE} Hz")
print(f"Segment duration   : {SEGMENT_DURATION_S} s")
print(f"Segment overlap    : {SEGMENT_OVERLAP}")
print(f"Labels: column index={LABELS_COLUMN_INDEX}, delimiter='{LABELS_DELIMITER}', header={LABELS_HAS_HEADER}")

---
## Step 3 — Scan audio files

This cell scans your audio folder, discovers all recording sites and files, and shows a summary of what will be analyzed.

**Site detection:**
- If your audio folder contains **subfolders**, each subfolder is treated as one recording site.
- If all audio files are **directly in the root folder**, the folder itself is treated as a single site.

**Datetime parsing:** timestamps are read from filenames using the prefix you configured. Files whose names do not match the expected pattern are still included — their date/time columns in results will be left blank.

In [7]:
#@title 🔍 Scan { display-mode: "form" }

import os
import re
import glob as _glob
from collections import defaultdict
from datetime import datetime

def parse_file_datetime(filename, prefix=''):
    name = os.path.splitext(os.path.basename(filename))[0]
    m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
    if m:
        return datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)),
                        int(m.group(4)), int(m.group(5)), int(m.group(6)))
    if prefix:
        m = re.search(rf'{re.escape(prefix)}_(\d{{8}})_(\d{{6}})', name)
    else:
        m = re.search(r'(\d{8})_(\d{6})', name)
    if m:
        d, t = m.group(1), m.group(2)
        return datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                        int(t[:2]), int(t[2:4]), int(t[4:6]))
    return None

def get_site_name(audio_path, input_dir):
    parent = os.path.normpath(os.path.dirname(audio_path))
    root   = os.path.normpath(input_dir)
    if parent == root:
        return os.path.basename(root)
    return os.path.basename(parent)

if not os.path.isdir(DRIVE_INPUT_DIR):
    raise FileNotFoundError(
        f"Audio folder not found: {DRIVE_INPUT_DIR}\n"
        "Please check the path in the Audio Input form (Step 2)."
    )

all_audio = sorted(
    _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.wav'),  recursive=True) +
    _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.flac'), recursive=True) +
    _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.mp3'),  recursive=True)
)

if FILTER_ENABLED and all_audio:
    _start = datetime.strptime(f"{FILTER_START_DATE} {FILTER_START_TIME or '00:00'}", '%Y-%m-%d %H:%M')
    _end   = datetime.strptime(f"{FILTER_END_DATE} {FILTER_END_TIME or '23:59'}", '%Y-%m-%d %H:%M')
    _n_before = len(all_audio)
    all_audio = [
        f for f in all_audio
        if (lambda dt: dt is None or _start <= dt <= _end)(parse_file_datetime(f, FILENAME_PREFIX))
    ]
    print(f"Date filter  : {_start.strftime('%Y-%m-%d %H:%M')} → {_end.strftime('%Y-%m-%d %H:%M')}")
    print(f"Files excluded by filter : {_n_before - len(all_audio)}")
    print()

if not all_audio:
    print(f"No audio files found in: {DRIVE_INPUT_DIR}")
    print("Supported formats: .wav, .flac, .mp3")
else:
    sites = defaultdict(list)
    no_datetime = []

    for f in all_audio:
        site = get_site_name(f, DRIVE_INPUT_DIR)
        dt   = parse_file_datetime(f, FILENAME_PREFIX)
        sites[site].append((f, dt))
        if dt is None:
            no_datetime.append(os.path.basename(f))

    print(f"Audio folder : {DRIVE_INPUT_DIR}")
    print(f"Total files  : {len(all_audio)}")
    print(f"Sites found  : {len(sites)}")
    print()

    for site_name, entries in sorted(sites.items()):
        dts = [dt for _, dt in entries if dt is not None]
        if dts:
            date_range = f"{min(dts).strftime('%Y-%m-%d %H:%M')} → {max(dts).strftime('%Y-%m-%d %H:%M')}"
        else:
            date_range = '(no timestamps parsed)'
        print(f"  {site_name}")
        print(f"    Files : {len(entries)}")
        print(f"    Range : {date_range}")

    if no_datetime:
        print()
        print(f"  WARNING: {len(no_datetime)} file(s) did not match the filename pattern:")
        for fn in no_datetime[:5]:
            print(f"    {fn}")
        if len(no_datetime) > 5:
            print(f"    ... and {len(no_datetime) - 5} more")
        if FILENAME_PREFIX:
            print(f"  Expected pattern: {FILENAME_PREFIX}_YYYYMMDD_HHMMSS.*")
        else:
            print(f"  Expected pattern: YYYYMMDD_HHMMSS.*")
        print("  These files will still be analyzed — date/time columns in results will be blank.")

    print()
    print("Scan complete. Continue to Step 4.")

---
## Step 4 — Load the model and species labels

This cell loads your model and reads the list of species (or other classes) it can detect.

**Labels file format:** a plain text file with one label per line, for example:
```
Phylodytes luteolus
Dendropsophus minutus
Background noise
```

The number of lines in the labels file must match the number of outputs of your model.

In [8]:
#@title 🧠 Load model and labels { display-mode: "form" }

import os
import numpy as np

_DELIMITERS = {"tab": "\t", "comma (,)": ",", "semicolon (;)":
               ";", "underscore (_)": "_"}
_labels_sep = _DELIMITERS.get(LABELS_DELIMITER, "\t")

if MODEL_SOURCE == 'huggingface':
    from huggingface_hub import hf_hub_download
    print(f'Downloading model from HuggingFace: {HF_REPO_ID} / {HF_MODEL_FILE} ...')
    model_path = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_MODEL_FILE)
    _labels_repo = HF_LABELS_REPO.strip() or HF_REPO_ID
    print(f'Downloading labels from HuggingFace: {_labels_repo} / {HF_LABELS_FILE} ...')
    labels_path = hf_hub_download(repo_id=_labels_repo, filename=HF_LABELS_FILE)
elif MODEL_SOURCE == 'google_drive':
    model_path  = DRIVE_MODEL_PATH
    labels_path = DRIVE_LABELS_PATH
else:
    raise ValueError(f"MODEL_SOURCE must be 'huggingface' or 'google_drive', got: {MODEL_SOURCE!r}")

if not os.path.exists(model_path):
    raise FileNotFoundError(
        f"Model file not found: {model_path}\n"
        "Please check the path in the Model form (Step 2)."
    )
if not os.path.exists(labels_path):
    raise FileNotFoundError(
        f"Labels file not found: {labels_path}\n"
        "Please check the path in the Model form (Step 2)."
    )

ext = os.path.splitext(model_path)[1].lower()
if ext == '.tflite':
    MODEL_TYPE = 'tflite'
elif ext == '.onnx':
    MODEL_TYPE = 'onnx'
else:
    raise ValueError(f"Unsupported model format: '{ext}'.\nThe model file must end in .tflite or .onnx")

MODEL_NAME = os.path.splitext(os.path.basename(model_path))[0]

if MODEL_TYPE == 'tflite':
    from ai_edge_litert.interpreter import Interpreter as TFLiteInterpreter
    model = TFLiteInterpreter(model_path=model_path)
    model.allocate_tensors()
    in_shape  = model.get_input_details()[0]['shape']
    out_shape = model.get_output_details()[0]['shape']
    print(f'TFLite model loaded.')
    print(f'  Input  shape : {in_shape}')
    print(f'  Output shape : {out_shape}')
elif MODEL_TYPE == 'onnx':
    import onnxruntime as ort
    model = ort.InferenceSession(model_path)
    in_shape  = model.get_inputs()[0].shape
    out_shape = model.get_outputs()[0].shape
    print(f'ONNX model loaded.')
    print(f'  Input  shape : {in_shape}')
    print(f'  Output shape : {out_shape}')

labels = []
with open(labels_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

if LABELS_HAS_HEADER and lines:
    lines = lines[1:]

for line in lines:
    line = line.strip()
    if not line:
        continue
    if _labels_sep in line:
        parts = line.split(_labels_sep)
        if LABELS_COLUMN_INDEX < len(parts):
            labels.append(parts[LABELS_COLUMN_INDEX].strip())
        else:
            print(f"  WARNING: column index {LABELS_COLUMN_INDEX} not found in line: {line!r} — skipping.")
    else:
        labels.append(line)

print(f'\nLabels loaded: {len(labels)} classes')
print(f'  Delimiter : {LABELS_DELIMITER}  |  Column index : {LABELS_COLUMN_INDEX}  |  Header skipped : {LABELS_HAS_HEADER}')
print(f'  First 5   : {labels[:5]}')
print(f'  Model name: {MODEL_NAME}')

---
## Step 5 — Check already completed analyses

This cell scans your results folder to see which audio files have **already been analyzed**. Those files will be **skipped** in the next step, so you can safely re-run the analysis without repeating work you've already done.

In [9]:
#@title 🔎 Check { display-mode: "form" }

import os
import re
import glob as _glob
from datetime import datetime

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

def parse_file_datetime(filename, prefix=''):
    name = os.path.splitext(os.path.basename(filename))[0]
    m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
    if m:
        return datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)),
                        int(m.group(4)), int(m.group(5)), int(m.group(6)))
    if prefix:
        m = re.search(rf'{re.escape(prefix)}_(\d{{8}})_(\d{{6}})', name)
    else:
        m = re.search(r'(\d{8})_(\d{6})', name)
    if m:
        d, t = m.group(1), m.group(2)
        return datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                        int(t[:2]), int(t[2:4]), int(t[4:6]))
    return None

def get_site_name(audio_path, input_dir):
    parent = os.path.normpath(os.path.dirname(audio_path))
    root   = os.path.normpath(input_dir)
    if parent == root:
        return os.path.basename(root)
    return os.path.basename(parent)

def get_result_path(audio_path, input_dir, results_dir, model_name, prefix=''):
    filename  = os.path.basename(audio_path)
    site_name = get_site_name(audio_path, input_dir)
    file_dt   = parse_file_datetime(filename, prefix)
    if file_dt is not None:
        dt_str    = file_dt.strftime('%Y%m%d_%H%M%S')
        result_fn = f"{site_name}_{dt_str}.{model_name}.results.txt"
    else:
        file_base = os.path.splitext(filename)[0]
        result_fn = f"{site_name}_{file_base}.{model_name}.results.txt"
    return os.path.join(results_dir, site_name, result_fn)

all_audio = sorted(
    _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.wav'),  recursive=True) +
    _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.flac'), recursive=True) +
    _glob.glob(os.path.join(DRIVE_INPUT_DIR, '**', '*.mp3'),  recursive=True)
)

if FILTER_ENABLED and all_audio:
    _start = datetime.strptime(f"{FILTER_START_DATE} {FILTER_START_TIME or '00:00'}", '%Y-%m-%d %H:%M')
    _end   = datetime.strptime(f"{FILTER_END_DATE} {FILTER_END_TIME or '23:59'}", '%Y-%m-%d %H:%M')
    all_audio = [
        f for f in all_audio
        if (lambda dt: dt is None or _start <= dt <= _end)(parse_file_datetime(f, FILENAME_PREFIX))
    ]

if TIME_FILTER_ENABLED and all_audio:
    from datetime import time as _dtime
    _ps = TIME_FILTER_START.split(':')
    _pe = TIME_FILTER_END.split(':')
    _ts = _dtime(int(_ps[0]), int(_ps[1]))
    _te = _dtime(int(_pe[0]), int(_pe[1]))
    def _tod_ok(f):
        dt = parse_file_datetime(f, FILENAME_PREFIX)
        if dt is None:
            return True
        t = dt.time()
        return (_ts <= t <= _te) if _ts <= _te else (t >= _ts or t <= _te)
    all_audio = [f for f in all_audio if _tod_ok(f)]

to_process   = [f for f in all_audio
                if not os.path.exists(
                    get_result_path(f, DRIVE_INPUT_DIR, DRIVE_RESULTS_DIR,
                                    MODEL_NAME, FILENAME_PREFIX))]
already_done = [f for f in all_audio if f not in to_process]

print(f'Results folder : {DRIVE_RESULTS_DIR}')
print(f'Model name     : {MODEL_NAME}')
if FILTER_ENABLED:
    print(f'Date filter    : {FILTER_START_DATE} {FILTER_START_TIME} → {FILTER_END_DATE} {FILTER_END_TIME}')
print()
print(f'Total audio files  : {len(all_audio)}')
print(f'Already analyzed   : {len(already_done)}')
print(f'Remaining to run   : {len(to_process)}')

if already_done:
    print()
    print('Already done (will be skipped):')
    for f in already_done:
        rp = get_result_path(f, DRIVE_INPUT_DIR, DRIVE_RESULTS_DIR,
                             MODEL_NAME, FILENAME_PREFIX)
        print(f"  {os.path.basename(f)}  →  {os.path.basename(rp)}")

if not to_process:
    print()
    print('All files have already been analyzed. Nothing to do!')
    print('If you want to re-analyze, delete the corresponding results files from your Drive first.')
else:
    print()
    print(f'Ready to analyze {len(to_process)} file(s). Proceed to Step 6.')

---
## Step 6 — Run the analysis and save results

This is the main analysis step. For each audio file:
1. The recording is split into short segments (e.g. 3 seconds each)
2. Each segment is fed to the model
3. Detections above the confidence threshold are kept
4. Results are saved immediately to your Google Drive

**Result files** are saved as:
```
DRIVE_RESULTS_DIR / SITE_NAME / audio_file.MODEL_NAME.results.txt
```

Each results file has columns: `site`, `date`, `time`, `start_time`, `end_time`, `label`, `confidence`

- `site` — the recording site name (subfolder name, or root folder name if no subfolders).
- `date` / `time` — timestamp of the detection segment, read from the filename.
- `start_time` / `end_time` — offset in seconds from the beginning of the audio file.

> Depending on the number of files and the length of recordings, this step can take a while. Progress is shown below.

In [11]:
#@title 🚀 Run { display-mode: "form" }
import csv
import re
import shutil
import time
import numpy as np
import librosa
from datetime import datetime, timedelta

def parse_file_datetime(filename, prefix=''):
    name = os.path.splitext(os.path.basename(filename))[0]
    m = re.search(r'(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})', name)
    if m:
        return datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)),
                        int(m.group(4)), int(m.group(5)), int(m.group(6)))
    if prefix:
        m = re.search(rf'{re.escape(prefix)}_(\d{{8}})_(\d{{6}})', name)
    else:
        m = re.search(r'(\d{8})_(\d{6})', name)
    if m:
        d, t = m.group(1), m.group(2)
        return datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                        int(t[:2]), int(t[2:4]), int(t[4:6]))
    return None

def get_site_name(audio_path, input_dir):
    parent = os.path.normpath(os.path.dirname(audio_path))
    root   = os.path.normpath(input_dir)
    if parent == root:
        return os.path.basename(root)
    return os.path.basename(parent)

def apply_activation(logits, activation, sensitivity=-1.0, bias=1.0):
    arr = np.array(logits, dtype=np.float32).flatten()
    if activation == 'sigmoid':
        transformed_bias = (bias - 1.0) * 10.0
        return 1.0 / (1.0 + np.exp(sensitivity * np.clip(arr + transformed_bias, -20, 20)))
    elif activation == 'softmax':
        arr = arr - arr.max()
        e = np.exp(arr)
        return e / e.sum()
    return arr

def run_model(model, model_type, audio_segment):
    seg = audio_segment.astype(np.float32)
    if model_type == 'tflite':
        in_det  = model.get_input_details()[0]
        out_det = model.get_output_details()[0]
        try:
            model.set_tensor(in_det['index'], seg.reshape(in_det['shape']))
        except ValueError:
            model.set_tensor(in_det['index'], seg.reshape(1, -1))
        model.invoke()
        return model.get_tensor(out_det['index']).flatten()
    else:
        input_name = model.get_inputs()[0].name
        try:
            logits = model.run(None, {input_name: seg.reshape(1, -1)})[0]
        except Exception:
            logits = model.run(None, {input_name: seg[np.newaxis, :]})[0]
        return np.array(logits).flatten()

def preprocess_audio(audio, sr):
    if FILTER_TYPE != 'none':
        from scipy.signal import butter, sosfilt
        nyq = sr / 2.0
        if FILTER_TYPE == 'lowpass':
            sos = butter(5, min(FILTER_HIGH_HZ, nyq - 1) / nyq, btype='low', output='sos')
        elif FILTER_TYPE == 'highpass':
            sos = butter(5, max(FILTER_LOW_HZ, 1) / nyq, btype='high', output='sos')
        elif FILTER_TYPE == 'bandpass':
            lo = max(FILTER_LOW_HZ, 1) / nyq
            hi = min(FILTER_HIGH_HZ, nyq - 1) / nyq
            sos = butter(5, [lo, hi], btype='band', output='sos')
        audio = sosfilt(sos, audio).astype(np.float32)
    if AUDIO_SPEED != 1.0:
        audio = librosa.effects.time_stretch(audio, rate=AUDIO_SPEED)
    return audio

segment_samples  = int(SEGMENT_DURATION_S * SAMPLE_RATE)
stride_samples   = max(1, int(segment_samples * (1 - SEGMENT_OVERLAP)))
def _in_time_window(file_dt, start_str, end_str):
    from datetime import time as _dtime
    if file_dt is None:
        return True
    t = file_dt.time()
    ps, pe = start_str.split(':'), end_str.split(':')
    t_start = _dtime(int(ps[0]), int(ps[1]))
    t_end   = _dtime(int(pe[0]), int(pe[1]))
    if t_start <= t_end:
        return t_start <= t <= t_end
    return t >= t_start or t <= t_end

total_detections = 0
total_start      = time.time()

if not to_process:
    print('No files to process. All analyses are already complete.')
else:
    print(f'Starting analysis of {len(to_process)} file(s) with model "{MODEL_NAME}"')
    print(f'Confidence threshold : {MIN_CONFIDENCE}')
    print(f'Top K                : {TOP_K}')
    print(f'Segment duration     : {SEGMENT_DURATION_S}s  |  Sample rate: {SAMPLE_RATE} Hz')
    if FILTER_TYPE != 'none' or AUDIO_SPEED != 1.0:
        _pp = []
        if FILTER_TYPE != 'none': _pp.append(f'filter={FILTER_TYPE}')
        if AUDIO_SPEED != 1.0:   _pp.append(f'speed={AUDIO_SPEED}x')
        print(f'Preprocessing        : {", ".join(_pp)}')
    print('=' * 65)

    for file_idx, audio_path in enumerate(to_process, 1):
        filename  = os.path.basename(audio_path)
        site_name = get_site_name(audio_path, DRIVE_INPUT_DIR)
        file_dt   = parse_file_datetime(filename, FILENAME_PREFIX)
        if TIME_FILTER_ENABLED and not _in_time_window(file_dt, TIME_FILTER_START, TIME_FILTER_END):
            print(f'[{file_idx}/{len(to_process)}] {filename}  — skipped (outside {TIME_FILTER_START}–{TIME_FILTER_END})')
            continue

        if file_dt is not None:
            dt_str    = file_dt.strftime('%Y%m%d_%H%M%S')
            result_fn = f"{site_name}_{dt_str}.{MODEL_NAME}.results.txt"
        else:
            file_base = os.path.splitext(filename)[0]
            result_fn = f"{site_name}_{file_base}.{MODEL_NAME}.results.txt"
            print(f"  WARNING: could not parse datetime from '{filename}' — using filename as fallback.")

        print(f'[{file_idx}/{len(to_process)}] {filename}  (site: {site_name})')

        os.makedirs('/content/audio_tmp', exist_ok=True)
        _local = os.path.join('/content/audio_tmp', os.path.basename(audio_path))
        shutil.copy2(audio_path, _local)
        try:
            audio, _ = librosa.load(_local, sr=SAMPLE_RATE, mono=True)
        except Exception as e:
            os.remove(_local)
            print(f'  ERROR loading audio: {e} — skipping.')
            continue
        os.remove(_local)

        audio = preprocess_audio(audio, SAMPLE_RATE)

        rows       = []
        file_start = time.time()
        n_segments = 0

        for start_samp in range(0, len(audio), stride_samples):
            seg = audio[start_samp:start_samp + segment_samples]
            if len(seg) < segment_samples * 0.5:
                continue
            if len(seg) < segment_samples:
                seg = np.pad(seg, (0, segment_samples - len(seg)))

            try:
                logits = run_model(model, MODEL_TYPE, seg)
            except Exception as e:
                print(f'  ERROR during inference at {start_samp/SAMPLE_RATE:.1f}s: {e}')
                continue

            n_segments += 1
            scores     = apply_activation(logits, ACTIVATION, SIGMOID_SENSITIVITY, SIGMOID_BIAS)
            offset_s   = start_samp / SAMPLE_RATE
            end_s      = offset_s + SEGMENT_DURATION_S

            _seg_hits = sorted(
                [(float(scores[i]), i) for i in range(min(len(scores), len(labels)))
                 if float(scores[i]) >= MIN_CONFIDENCE],
                reverse=True
            )[:TOP_K]
            for score, idx in _seg_hits:
                if file_dt is not None:
                    date_str = file_dt.strftime('%Y-%m-%d')
                    time_str = file_dt.strftime('%H:%M:%S')
                else:
                    date_str = ''
                    time_str = ''
                site_lat, site_lon = _latlon_map.get(site_name, ('', ''))
                rows.append([site_name, site_lat, site_lon, date_str, time_str,
                                f'{offset_s:.1f}', f'{end_s:.1f}',
                                labels[idx], f'{float(score):.4f}'])

        elapsed     = time.time() - file_start
        per_seg     = elapsed / n_segments if n_segments else 0
        audio_dur_s = len(audio) / SAMPLE_RATE

        site_dir    = os.path.join(DRIVE_RESULTS_DIR, site_name)
        os.makedirs(site_dir, exist_ok=True)
        result_path = os.path.join(site_dir, result_fn)

        with open(result_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['site', 'lat', 'lon', 'date', 'time', 'start_time', 'end_time', 'label', 'confidence'])
            writer.writerows(rows)

        total_detections += len(rows)
        print(f'  -> {len(rows)} detection(s)  |  {n_segments} segments  |  '
              f'{elapsed:.1f}s ({per_seg*1000:.0f}ms/seg, {audio_dur_s/elapsed:.1f}x realtime)  '
              f'→  {result_fn}')

    total_elapsed = time.time() - total_start
    print()
    print('=' * 65)
    print(f'Analysis complete!')
    print(f'  Files analyzed   : {len(to_process)}')
    print(f'  Total detections : {total_detections}')
    print(f'  Total time       : {total_elapsed:.1f}s')
    print(f'  Results saved to : {DRIVE_RESULTS_DIR}')

---
## Merge Results into a Single CSV

The Run step above saves one result file per recording (so an interrupted run can resume). This step
combines all of those files into a **single CSV** on your Google Drive:
```
RESULTS_FOLDER / MODEL_NAME.merged.results.csv
```
Columns: `stream_id`, `site`, `lat`, `lon`, `date`, `time`, `utc_offset`, `start_time`, `end_time`, `label`, `confidence`

Load this file directly in the **Classification Results Analyzer** notebook by pointing its
*Results folder* setting at this `.merged.results.csv` file.

> The individual per-recording files are **not** deleted — you can safely re-run this step at any time.

In [ ]:
#@title 🧩 Merge all results into a single CSV { display-mode: "form" }

#@markdown Combines every per-recording result file in the results folder into one CSV,
#@markdown ready to load in the **Classification Results Analyzer** notebook.
#@markdown The individual per-recording files are kept so a crashed run can still resume.

import os
import csv
import glob

MERGED_CSV_PATH = os.path.join(DRIVE_RESULTS_DIR, f'{MODEL_NAME}.merged.results.csv')

# Find every per-recording result file written by the Run step (searched recursively, one per site folder).
_result_files = sorted(glob.glob(
    os.path.join(DRIVE_RESULTS_DIR, '**', f'*.{MODEL_NAME}.results.txt'), recursive=True))

if not _result_files:
    print(f'No result files found in: {DRIVE_RESULTS_DIR}')
    print('Run the analysis step first to generate the per-recording results.')
else:
    # Stream file-by-file so we never hold every result in memory at once.
    _header  = None
    _n_files = 0
    _n_rows  = 0
    with open(MERGED_CSV_PATH, 'w', newline='', encoding='utf-8') as _out:
        _writer = csv.writer(_out)
        for _fp in _result_files:
            try:
                with open(_fp, 'r', newline='', encoding='utf-8') as _in:
                    _rows = list(csv.reader(_in))
            except Exception as _e:
                print(f'  WARNING: could not read {os.path.basename(_fp)}: {_e}')
                continue
            if not _rows:
                continue
            _file_header, _data = _rows[0], _rows[1:]
            if _header is None:
                _header = _file_header
                _writer.writerow(_header)
            _writer.writerows(_data)
            _n_files += 1
            _n_rows  += len(_data)

    print('Merge complete!')
    print(f'  Result files merged : {_n_files}')
    print(f'  Total detections    : {_n_rows}')
    print(f'  Merged CSV saved to : {MERGED_CSV_PATH}')